In [1]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent

EVALUATION_SET_PATH = (
    PROJECT_ROOT / "data" / "outputs" / "evaluation_set.json"
)

VECTOR_INDEX_PATH = (
    PROJECT_ROOT / "data" / "vector_store" / "quant_research.faiss"
)

METADATA_PATH = (
    PROJECT_ROOT / "data" / "vector_store" / "quant_research_metadata.json"
)

print("Evaluation set:", EVALUATION_SET_PATH)
print("FAISS index:", VECTOR_INDEX_PATH)
print("Metadata:", METADATA_PATH)

Evaluation set: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\evaluation_set.json
FAISS index: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\vector_store\quant_research.faiss
Metadata: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\vector_store\quant_research_metadata.json


In [2]:
with open(EVALUATION_SET_PATH, "r", encoding="utf-8") as f:
    evaluation_set = json.load(f)

print("Evaluation questions:", len(evaluation_set))

for item in evaluation_set[:5]:
    print(
        item["query_id"],
        "→",
        item["category"],
        "→",
        len(item["relevant_chunks"]),
        "gold chunk(s)"
    )

Evaluation questions: 26
SEM01 → semantic_factual → 1 gold chunk(s)
SEM02 → semantic_factual → 1 gold chunk(s)
SEM03 → semantic_factual → 1 gold chunk(s)
SEM04 → semantic_factual → 1 gold chunk(s)
SEM05 → semantic_factual → 1 gold chunk(s)


In [3]:
import faiss

# Load dense vector index
faiss_index = faiss.read_index(str(VECTOR_INDEX_PATH))

# Load chunk metadata
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    chunk_metadata = json.load(f)

print("FAISS vectors:", faiss_index.ntotal)
print("Metadata records:", len(chunk_metadata))
print("Vector dimension:", faiss_index.d)

FAISS vectors: 1141
Metadata records: 1141
Vector dimension: 384


In [4]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

c:\Users\MD RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3346.59it/s]


Embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


C:\Users\MD RAJA\AppData\Local\Temp\ipykernel_28160\2425308492.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [5]:
import numpy as np

def embed_query(query: str) -> np.ndarray:
    vector = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    return vector.astype("float32")

# Quick QA
test_vector = embed_query(evaluation_set[0]["query"])

print("Query:", evaluation_set[0]["query"])
print("Query vector shape:", test_vector.shape)

Query: What developments have enabled the rise of quantitative investors (quants), and how do they differ from traditional discretionary investors?
Query vector shape: (1, 384)


In [6]:
query_vector = embed_query(evaluation_set[0]["query"])

scores, indices = faiss_index.search(query_vector, 10)

print("Top-10 FAISS results:")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = chunk_metadata[idx]
    print(
        f"{rank}. {chunk['chunk_id']} | "
        f"score={score:.4f} | "
        f"source={chunk['source_file']}"
    )

Top-10 FAISS results:
1. chunk_f0d41995c19a42f8 | score=0.8280 | source=ssrn-5231379.pdf
2. chunk_e9dbc9860c200c4a | score=0.7912 | source=ssrn-5231379.pdf
3. chunk_a449942ec7fbfc65 | score=0.7863 | source=ssrn-5231379.pdf
4. chunk_3a61830a8459dfa3 | score=0.7712 | source=ssrn-5231379.pdf
5. chunk_005af18671e7dab4 | score=0.7588 | source=ssrn-5231379.pdf
6. chunk_424b6cfa3bb0365e | score=0.7565 | source=ssrn-5231379.pdf
7. chunk_063dd6f2f3ba08b5 | score=0.7557 | source=ssrn-5231379.pdf
8. chunk_8b24fd6f214db44c | score=0.7548 | source=ssrn-5231379.pdf
9. chunk_04f1f4faf8f98e4b | score=0.7533 | source=ssrn-5231379.pdf
10. chunk_53c67217c9f61858 | score=0.7501 | source=ssrn-5231379.pdf


In [7]:
from math import log2

gold_chunks = {
    item["chunk_id"]: item["relevance"]
    for item in evaluation_set[0]["relevant_chunks"]
}

retrieved_ids = [
    chunk_metadata[idx]["chunk_id"]
    for idx in indices[0]
]

# Recall@K
def recall_at_k(retrieved, gold, k):
    return int(any(chunk_id in gold for chunk_id in retrieved[:k]))

# MRR
def reciprocal_rank(retrieved, gold):
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in gold:
            return 1.0 / rank
    return 0.0

# nDCG@K
def ndcg_at_k(retrieved, gold, k):
    dcg = 0.0
    for rank, chunk_id in enumerate(retrieved[:k], start=1):
        relevance = gold.get(chunk_id, 0)
        dcg += relevance / log2(rank + 1)

    ideal = sorted(gold.values(), reverse=True)[:k]
    idcg = sum(
        relevance / log2(rank + 1)
        for rank, relevance in enumerate(ideal, start=1)
    )

    return dcg / idcg if idcg > 0 else 0.0


print("Gold chunks:", list(gold_chunks.keys()))
print("Retrieved rank:", retrieved_ids.index(list(gold_chunks.keys())[0]) + 1)

print("Recall@1 :", recall_at_k(retrieved_ids, gold_chunks, 1))
print("Recall@3 :", recall_at_k(retrieved_ids, gold_chunks, 3))
print("Recall@5 :", recall_at_k(retrieved_ids, gold_chunks, 5))
print("Recall@10:", recall_at_k(retrieved_ids, gold_chunks, 10))
print("MRR      :", reciprocal_rank(retrieved_ids, gold_chunks))
print("nDCG@10  :", round(ndcg_at_k(retrieved_ids, gold_chunks, 10), 4))

Gold chunks: ['chunk_e9dbc9860c200c4a']
Retrieved rank: 2
Recall@1 : 0
Recall@3 : 1
Recall@5 : 1
Recall@10: 1
MRR      : 0.5
nDCG@10  : 0.6309


In [8]:
faiss_results = []

for item in evaluation_set:
    query_vector = embed_query(item["query"])
    scores, indices = faiss_index.search(query_vector, 10)

    retrieved_ids = [
        chunk_metadata[idx]["chunk_id"]
        for idx in indices[0]
    ]

    gold_chunks = {
        x["chunk_id"]: x["relevance"]
        for x in item["relevant_chunks"]
    }

    faiss_results.append({
        "query_id": item["query_id"],
        "category": item["category"],
        "Recall@1": recall_at_k(retrieved_ids, gold_chunks, 1),
        "Recall@3": recall_at_k(retrieved_ids, gold_chunks, 3),
        "Recall@5": recall_at_k(retrieved_ids, gold_chunks, 5),
        "Recall@10": recall_at_k(retrieved_ids, gold_chunks, 10),
        "MRR": reciprocal_rank(retrieved_ids, gold_chunks),
        "nDCG@10": ndcg_at_k(retrieved_ids, gold_chunks, 10),
    })

print("FAISS evaluation completed:", len(faiss_results), "queries")
print(faiss_results[:3])

FAISS evaluation completed: 26 queries
[{'query_id': 'SEM01', 'category': 'semantic_factual', 'Recall@1': 0, 'Recall@3': 1, 'Recall@5': 1, 'Recall@10': 1, 'MRR': 0.5, 'nDCG@10': 0.6309297535714574}, {'query_id': 'SEM02', 'category': 'semantic_factual', 'Recall@1': 1, 'Recall@3': 1, 'Recall@5': 1, 'Recall@10': 1, 'MRR': 1.0, 'nDCG@10': 1.0}, {'query_id': 'SEM03', 'category': 'semantic_factual', 'Recall@1': 0, 'Recall@3': 0, 'Recall@5': 1, 'Recall@10': 1, 'MRR': 0.25, 'nDCG@10': 0.4306765580733931}]


In [9]:
import pandas as pd

faiss_df = pd.DataFrame(faiss_results)

faiss_summary = faiss_df[
    ["Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR", "nDCG@10"]
].mean()

print("FAISS Dense Retrieval — Overall Performance")
print("=" * 50)

for metric, value in faiss_summary.items():
    print(f"{metric:10s}: {value:.4f}")

FAISS Dense Retrieval — Overall Performance
Recall@1  : 0.1538
Recall@3  : 0.3846
Recall@5  : 0.5385
Recall@10 : 0.5769
MRR       : 0.2949
nDCG@10   : 0.3506


In [10]:
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"

print("Vector store files:")
for path in sorted(VECTOR_STORE_DIR.iterdir()):
    print("-", path.name)

Vector store files:
- graph_aware_context.json
- hybrid_retrieval_config.json
- quant_research.faiss
- quant_research_bm25.pkl
- quant_research_bm25_ids.json
- quant_research_knowledge_graph.graphml
- quant_research_knowledge_graph.json
- quant_research_metadata.json
- rag_last_response.json
- reranking_config.json


In [11]:
import pickle

BM25_PATH = VECTOR_STORE_DIR / "quant_research_bm25.pkl"
BM25_IDS_PATH = VECTOR_STORE_DIR / "quant_research_bm25_ids.json"

with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)

with open(BM25_IDS_PATH, "r", encoding="utf-8") as f:
    bm25_chunk_ids = json.load(f)

print("BM25 corpus size:", len(bm25_chunk_ids))
print("BM25 index loaded:", bm25 is not None)
print("First 5 BM25 chunk IDs:", bm25_chunk_ids[:5])

BM25 corpus size: 1132
BM25 index loaded: True
First 5 BM25 chunk IDs: ['chunk_634b3e042dffad81', 'chunk_e9dbc9860c200c4a', 'chunk_f0d41995c19a42f8', 'chunk_09712835319a6dd6', 'chunk_d9600aaaec5dcae2']


In [12]:
query = evaluation_set[0]["query"]

# BM25 returns scores for every indexed document
bm25_scores = bm25.get_scores(query.split())

# Get top 10 BM25 results
top_indices = np.argsort(bm25_scores)[::-1][:10]

bm25_retrieved_ids = [
    bm25_chunk_ids[idx]
    for idx in top_indices
]

print("Top-10 BM25 results:")
for rank, (idx, chunk_id) in enumerate(
    zip(top_indices, bm25_retrieved_ids), start=1
):
    print(
        f"{rank}. {chunk_id} | "
        f"score={bm25_scores[idx]:.4f}"
    )

Top-10 BM25 results:
1. chunk_e9dbc9860c200c4a | score=22.9346
2. chunk_f0d41995c19a42f8 | score=22.3773
3. chunk_a449942ec7fbfc65 | score=21.2543
4. chunk_005af18671e7dab4 | score=18.5810
5. chunk_424b6cfa3bb0365e | score=17.7213
6. chunk_0b2bc0798ea175d1 | score=15.9947
7. chunk_fa626fbec884d89e | score=15.4049
8. chunk_063dd6f2f3ba08b5 | score=14.7665
9. chunk_09712835319a6dd6 | score=14.3583
10. chunk_d43c521781ebc012 | score=13.9392


In [13]:
gold_chunks = {
    item["chunk_id"]: item["relevance"]
    for item in evaluation_set[0]["relevant_chunks"]
}

print("Gold chunks:", list(gold_chunks.keys()))
print(
    "Retrieved rank:",
    bm25_retrieved_ids.index(list(gold_chunks.keys())[0]) + 1
)

print("Recall@1 :", recall_at_k(bm25_retrieved_ids, gold_chunks, 1))
print("Recall@3 :", recall_at_k(bm25_retrieved_ids, gold_chunks, 3))
print("Recall@5 :", recall_at_k(bm25_retrieved_ids, gold_chunks, 5))
print("Recall@10:", recall_at_k(bm25_retrieved_ids, gold_chunks, 10))
print("MRR      :", reciprocal_rank(bm25_retrieved_ids, gold_chunks))
print("nDCG@10  :", round(
    ndcg_at_k(bm25_retrieved_ids, gold_chunks, 10), 4
))

Gold chunks: ['chunk_e9dbc9860c200c4a']
Retrieved rank: 1
Recall@1 : 1
Recall@3 : 1
Recall@5 : 1
Recall@10: 1
MRR      : 1.0
nDCG@10  : 1.0


In [14]:
bm25_results = []

for item in evaluation_set:
    query = item["query"]

    bm25_scores = bm25.get_scores(query.split())
    top_indices = np.argsort(bm25_scores)[::-1][:10]

    retrieved_ids = [
        bm25_chunk_ids[idx]
        for idx in top_indices
    ]

    gold_chunks = {
        x["chunk_id"]: x["relevance"]
        for x in item["relevant_chunks"]
    }

    bm25_results.append({
        "query_id": item["query_id"],
        "category": item["category"],
        "Recall@1": recall_at_k(retrieved_ids, gold_chunks, 1),
        "Recall@3": recall_at_k(retrieved_ids, gold_chunks, 3),
        "Recall@5": recall_at_k(retrieved_ids, gold_chunks, 5),
        "Recall@10": recall_at_k(retrieved_ids, gold_chunks, 10),
        "MRR": reciprocal_rank(retrieved_ids, gold_chunks),
        "nDCG@10": ndcg_at_k(retrieved_ids, gold_chunks, 10),
    })

print("BM25 evaluation completed:", len(bm25_results), "queries")

BM25 evaluation completed: 26 queries


In [15]:
bm25_df = pd.DataFrame(bm25_results)

bm25_summary = bm25_df[
    ["Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR", "nDCG@10"]
].mean()

print("BM25 Lexical Retrieval — Overall Performance")
print("=" * 50)

for metric, value in bm25_summary.items():
    print(f"{metric:10s}: {value:.4f}")

BM25 Lexical Retrieval — Overall Performance
Recall@1  : 0.1154
Recall@3  : 0.2308
Recall@5  : 0.3077
Recall@10 : 0.4231
MRR       : 0.1897
nDCG@10   : 0.2399


In [16]:
RRF_K = 60

def rrf_fuse(dense_ids, bm25_ids, k=RRF_K):
    rrf_scores = {}

    for rank, chunk_id in enumerate(dense_ids, start=1):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank)

    for rank, chunk_id in enumerate(bm25_ids, start=1):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank)

    ranked = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked


# SEM01
query = evaluation_set[0]["query"]

# Dense retrieval
query_vector = embed_query(query)
dense_scores, dense_indices = faiss_index.search(query_vector, 10)

dense_ids = [
    chunk_metadata[idx]["chunk_id"]
    for idx in dense_indices[0]
]

# BM25 retrieval
bm25_scores = bm25.get_scores(query.split())
bm25_indices = np.argsort(bm25_scores)[::-1][:10]

bm25_ids = [
    bm25_chunk_ids[idx]
    for idx in bm25_indices
]

# RRF fusion
hybrid_ranked = rrf_fuse(dense_ids, bm25_ids)

print("Top-10 RRF Hybrid results:")
for rank, (chunk_id, score) in enumerate(hybrid_ranked[:10], start=1):
    print(f"{rank}. {chunk_id} | RRF score={score:.6f}")

Top-10 RRF Hybrid results:
1. chunk_f0d41995c19a42f8 | RRF score=0.032522
2. chunk_e9dbc9860c200c4a | RRF score=0.032522
3. chunk_a449942ec7fbfc65 | RRF score=0.031746
4. chunk_005af18671e7dab4 | RRF score=0.031010
5. chunk_424b6cfa3bb0365e | RRF score=0.030536
6. chunk_063dd6f2f3ba08b5 | RRF score=0.029631
7. chunk_3a61830a8459dfa3 | RRF score=0.015625
8. chunk_0b2bc0798ea175d1 | RRF score=0.015152
9. chunk_fa626fbec884d89e | RRF score=0.014925
10. chunk_8b24fd6f214db44c | RRF score=0.014706


In [17]:
hybrid_results = []

for item in evaluation_set:
    query = item["query"]

    # Dense retrieval
    query_vector = embed_query(query)
    _, dense_indices = faiss_index.search(query_vector, 10)

    dense_ids = [
        chunk_metadata[idx]["chunk_id"]
        for idx in dense_indices[0]
    ]

    # BM25 retrieval
    bm25_scores = bm25.get_scores(query.split())
    bm25_indices = np.argsort(bm25_scores)[::-1][:10]

    bm25_ids = [
        bm25_chunk_ids[idx]
        for idx in bm25_indices
    ]

    # RRF fusion
    hybrid_ranked = rrf_fuse(dense_ids, bm25_ids)
    retrieved_ids = [x[0] for x in hybrid_ranked[:10]]

    gold_chunks = {
        x["chunk_id"]: x["relevance"]
        for x in item["relevant_chunks"]
    }

    hybrid_results.append({
        "query_id": item["query_id"],
        "category": item["category"],
        "Recall@1": recall_at_k(retrieved_ids, gold_chunks, 1),
        "Recall@3": recall_at_k(retrieved_ids, gold_chunks, 3),
        "Recall@5": recall_at_k(retrieved_ids, gold_chunks, 5),
        "Recall@10": recall_at_k(retrieved_ids, gold_chunks, 10),
        "MRR": reciprocal_rank(retrieved_ids, gold_chunks),
        "nDCG@10": ndcg_at_k(retrieved_ids, gold_chunks, 10),
    })

print("RRF Hybrid evaluation completed:", len(hybrid_results), "queries")

RRF Hybrid evaluation completed: 26 queries


In [18]:
hybrid_df = pd.DataFrame(hybrid_results)

hybrid_summary = hybrid_df[
    ["Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR", "nDCG@10"]
].mean()

print("RRF Hybrid Retrieval — Overall Performance")
print("=" * 50)

for metric, value in hybrid_summary.items():
    print(f"{metric:10s}: {value:.4f}")

RRF Hybrid Retrieval — Overall Performance
Recall@1  : 0.1154
Recall@3  : 0.3462
Recall@5  : 0.5000
Recall@10 : 0.5769
MRR       : 0.2540
nDCG@10   : 0.3274


In [19]:
comparison_df = pd.DataFrame({
    "FAISS": faiss_summary,
    "BM25": bm25_summary,
    "RRF Hybrid": hybrid_summary
})

print("FAISS vs BM25 vs RRF Hybrid")
print("=" * 65)
print(comparison_df.round(4))

FAISS vs BM25 vs RRF Hybrid
            FAISS    BM25  RRF Hybrid
Recall@1   0.1538  0.1154      0.1154
Recall@3   0.3846  0.2308      0.3462
Recall@5   0.5385  0.3077      0.5000
Recall@10  0.5769  0.4231      0.5769
MRR        0.2949  0.1897      0.2540
nDCG@10    0.3506  0.2399      0.3274


Inference
- FAISS is the strongest retrieval baseline on this 26-query benchmark, outperforming both BM25 and RRF Hybrid across the reported metrics.
- BM25 alone is weaker, but it still contributes lexical matching that dense retrieval may miss.
- RRF Hybrid improves over BM25 but does not surpass FAISS: Recall@10 remains 0.5769, equal to FAISS, while MRR and nDCG@10 are lower.
- Therefore, hybrid fusion has not demonstrated a gain yet; the next Cross-Encoder stage is important because it can improve the ranking of the retrieved candidates rather than simply combining their rankings.

In [20]:
rrf_gain_vs_faiss = hybrid_summary - faiss_summary
rrf_gain_vs_bm25 = hybrid_summary - bm25_summary

gain_df = pd.DataFrame({
    "RRF vs FAISS": rrf_gain_vs_faiss,
    "RRF vs BM25": rrf_gain_vs_bm25
})

print("RRF Hybrid Gain Analysis")
print("=" * 60)
print(gain_df.round(4))

RRF Hybrid Gain Analysis
           RRF vs FAISS  RRF vs BM25
Recall@1        -0.0385       0.0000
Recall@3        -0.0385       0.1154
Recall@5        -0.0385       0.1923
Recall@10        0.0000       0.1538
MRR             -0.0409       0.0643
nDCG@10         -0.0232       0.0875


Inference
- RRF vs FAISS: RRF performs worse at Recall@1–5, MRR, and nDCG@10; it only matches FAISS at Recall@10.
- RRF vs BM25: RRF clearly improves retrieval, especially Recall@5 (+19.23 percentage points) and Recall@10 (+15.38 pp).
- Conclusion: RRF successfully strengthens BM25, but on this benchmark dense retrieval remains stronger than the current hybrid fusion.
- This makes the next stage important: Cross-Encoder Reranking should determine whether better ordering of the hybrid candidates improves the final retrieval quality.

In [21]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Cross-Encoder:", RERANKER_MODEL_NAME)
print("Reranker loaded: READY")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2624.83it/s]


Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
Reranker loaded: READY


In [22]:
rerank_results = []

for item in evaluation_set:
    query = item["query"]

    # Dense retrieval
    query_vector = embed_query(query)
    _, dense_indices = faiss_index.search(query_vector, 10)

    dense_ids = [
        chunk_metadata[idx]["chunk_id"]
        for idx in dense_indices[0]
    ]

    # BM25 retrieval
    bm25_scores = bm25.get_scores(query.split())
    bm25_indices = np.argsort(bm25_scores)[::-1][:10]

    bm25_ids = [
        bm25_chunk_ids[idx]
        for idx in bm25_indices
    ]

    # RRF Hybrid
    hybrid_ranked = rrf_fuse(dense_ids, bm25_ids)
    hybrid_ids = [x[0] for x in hybrid_ranked[:10]]

    # Build query-document pairs
    chunk_lookup = {
        x["chunk_id"]: x
        for x in chunk_metadata
    }

    pairs = [
        [query, chunk_lookup[chunk_id]["retrieval_text"]]
        for chunk_id in hybrid_ids
    ]

    # Cross-Encoder scores
    scores = reranker.predict(pairs)

    # Rerank
    ranked = sorted(
        zip(hybrid_ids, scores),
        key=lambda x: x[1],
        reverse=True
    )

    reranked_ids = [x[0] for x in ranked]

    gold_chunks = {
        x["chunk_id"]: x["relevance"]
        for x in item["relevant_chunks"]
    }

    rerank_results.append({
        "query_id": item["query_id"],
        "category": item["category"],
        "Recall@1": recall_at_k(reranked_ids, gold_chunks, 1),
        "Recall@3": recall_at_k(reranked_ids, gold_chunks, 3),
        "Recall@5": recall_at_k(reranked_ids, gold_chunks, 5),
        "Recall@10": recall_at_k(reranked_ids, gold_chunks, 10),
        "MRR": reciprocal_rank(reranked_ids, gold_chunks),
        "nDCG@10": ndcg_at_k(reranked_ids, gold_chunks, 10),
    })

print("Cross-Encoder evaluation completed:", len(rerank_results), "queries")

Cross-Encoder evaluation completed: 26 queries


In [23]:
print("Cross-Encoder evaluation completed:", len(rerank_results), "queries")

Cross-Encoder evaluation completed: 26 queries


In [24]:
rerank_df = pd.DataFrame(rerank_results)

rerank_summary = rerank_df[
    ["Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR", "nDCG@10"]
].mean()

print("Cross-Encoder Reranking — Overall Performance")
print("=" * 55)

for metric, value in rerank_summary.items():
    print(f"{metric:10s}: {value:.4f}")

Cross-Encoder Reranking — Overall Performance
Recall@1  : 0.2692
Recall@3  : 0.3846
Recall@5  : 0.5385
Recall@10 : 0.5769
MRR       : 0.3702
nDCG@10   : 0.4066


In [25]:
comparison_rerank_df = pd.DataFrame({
    "RRF Hybrid": hybrid_summary,
    "Cross-Encoder": rerank_summary
})

comparison_rerank_df["Gain"] = (
    comparison_rerank_df["Cross-Encoder"]
    - comparison_rerank_df["RRF Hybrid"]
)

print("RRF Hybrid vs Cross-Encoder")
print("=" * 65)
print(comparison_rerank_df.round(4))

RRF Hybrid vs Cross-Encoder
           RRF Hybrid  Cross-Encoder    Gain
Recall@1       0.1154         0.2692  0.1538
Recall@3       0.3462         0.3846  0.0385
Recall@5       0.5000         0.5385  0.0385
Recall@10      0.5769         0.5769  0.0000
MRR            0.2540         0.3702  0.1162
nDCG@10        0.3274         0.4066  0.0792


Cross-Encoder clearly improves ranking over RRF: MRR increases from 0.2540 → 0.3702 (+0.1162) and nDCG@10 from 0.3274 → 0.4066 (+0.0792).
Largest improvement is Recall@1: +0.1538, showing the relevant chunk is much more frequently moved to the top position.
Recall@10 is unchanged (0.5769): the Cross-Encoder does not retrieve additional candidates; it reorders the existing top-10 candidates.
Therefore, the architecture is behaving as intended: RRF generates a diverse candidate set, while the Cross-Encoder improves the ranking/precision of those candidates.

In [26]:
category_summary = (
    rerank_df
    .groupby("category")[
        ["Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR", "nDCG@10"]
    ]
    .mean()
    .round(4)
)

print("Cross-Encoder Performance by Query Category")
print("=" * 75)
print(category_summary)

Cross-Encoder Performance by Query Category
                       Recall@1  Recall@3  Recall@5  Recall@10     MRR  \
category                                                                 
cross_document           1.0000       1.0    1.0000     1.0000  1.0000   
figure_based             0.0000       0.0    0.0000     0.5000  0.0625   
formula_quantitative     0.0000       0.0    0.0000     0.0000  0.0000   
multi_chunk_reasoning    0.0000       0.0    1.0000     1.0000  0.2500   
semantic_factual         0.4000       0.6    0.7000     0.7000  0.5250   
table_based              0.3333       0.5    0.8333     0.8333  0.5000   

                       nDCG@10  
category                        
cross_document          0.6131  
figure_based            0.1577  
formula_quantitative    0.0000  
multi_chunk_reasoning   0.4575  
semantic_factual        0.5693  
table_based             0.5820  


Semantic and table queries benefit most: Recall@10 reaches 0.70 for semantic queries and 0.8333 for table queries.
Cross-document retrieval performs perfectly: Recall@1–10 and MRR are 1.0 on the single cross-document query.
Formula queries remain a major weakness: all retrieval metrics are 0, indicating the current retrieval/reranking pipeline is not finding the frozen gold formula chunks.
Figure retrieval is also weak: Recall@10 is only 0.50, suggesting multimodal evidence retrieval needs further improvement.
Overall, the Cross-Encoder improves ranking, but formula and figure retrieval are the main areas to investigate next, especially because the benchmark intentionally tests these modalities

In [27]:
ce_vs_faiss = pd.DataFrame({
    "FAISS": faiss_summary,
    "Cross-Encoder": rerank_summary
})

ce_vs_faiss["Gain"] = (
    ce_vs_faiss["Cross-Encoder"]
    - ce_vs_faiss["FAISS"]
)

print("FAISS vs Cross-Encoder")
print("=" * 65)
print(ce_vs_faiss.round(4))

FAISS vs Cross-Encoder
            FAISS  Cross-Encoder    Gain
Recall@1   0.1538         0.2692  0.1154
Recall@3   0.3846         0.3846  0.0000
Recall@5   0.5385         0.5385  0.0000
Recall@10  0.5769         0.5769  0.0000
MRR        0.2949         0.3702  0.0753
nDCG@10    0.3506         0.4066  0.0560


ross-Encoder improves over FAISS mainly in ranking quality: MRR increases from 0.2949 → 0.3702 (+0.0753) and nDCG@10 from 0.3506 → 0.4066 (+0.0560).
Recall@1 improves by +0.1154, meaning the relevant chunk is more often ranked first.
Recall@3, @5, and @10 remain unchanged, confirming that reranking improves the ordering of existing candidates rather than expanding retrieval coverage.
Overall, the Cross-Encoder provides a better final ranking than both FAISS and RRF, while the retrieval candidate pool remains the same.

In [28]:
GRAPH_CONTEXT_PATH = (
    VECTOR_STORE_DIR / "graph_aware_context.json"
)

with open(GRAPH_CONTEXT_PATH, "r", encoding="utf-8") as f:
    graph_context = json.load(f)

print("Graph-aware context loaded:", len(graph_context))

if graph_context:
    print("\nFirst record keys:")
    print(graph_context[0].keys())

Graph-aware context loaded: 7

First record keys:
dict_keys(['vector_id', 'chunk_id', 'document_id', 'source_file', 'chunk_index', 'chunk_type', 'text', 'retrieval_text', 'token_count', 'section_path', 'pages', 'element_ids', 'content_types', 'figure_ids', 'table_ids', 'formula_ids', 'image_paths', 'evidence', 'selection_reason'])


In [29]:
# Build lookup for graph-aware context
graph_chunk_ids = {
    record["chunk_id"]
    for record in graph_context
}

graph_eval = []

for item in evaluation_set:
    gold_ids = {
        x["chunk_id"]
        for x in item["relevant_chunks"]
    }

    # Find the Cross-Encoder result for this query
    reranked_row = rerank_df[
        rerank_df["query_id"] == item["query_id"]
    ]

    # Graph artifact contains selected context records
    selected_ids = list(graph_chunk_ids)

    gold_in_graph = gold_ids.intersection(selected_ids)

    graph_eval.append({
        "query_id": item["query_id"],
        "category": item["category"],
        "gold_chunks": len(gold_ids),
        "graph_context_chunks": len(selected_ids),
        "relevant_chunks_in_graph": len(gold_in_graph),
        "graph_coverage": (
            len(gold_in_graph) / len(gold_ids)
            if gold_ids else 0
        )
    })

graph_eval_df = pd.DataFrame(graph_eval)

print("Graph-Aware Context Evaluation")
print("=" * 60)
print(
    graph_eval_df[
        ["graph_coverage"]
    ].mean().round(4)
)

Graph-Aware Context Evaluation
graph_coverage    0.0769
dtype: float64


In [30]:
for i, record in enumerate(graph_context, start=1):
    print(
        f"{i}. {record['chunk_id']} | "
        f"{record['selection_reason']} | "
        f"{record['source_file']} | "
        f"pages={record['pages']}"
    )

1. chunk_634b3e042dffad81 | reranked | ssrn-5231379.pdf | pages=[1]
2. chunk_e9dbc9860c200c4a | reranked | ssrn-5231379.pdf | pages=[2]
3. chunk_f0d41995c19a42f8 | reranked | ssrn-5231379.pdf | pages=[2, 3]
4. chunk_09712835319a6dd6 | reranked | ssrn-5231379.pdf | pages=[3, 4]
5. chunk_d9600aaaec5dcae2 | reranked | ssrn-5231379.pdf | pages=[4]
6. chunk_f1c8dbdb6d16fb1b | graph_neighbor | ssrn-5231379.pdf | pages=[5]
7. chunk_424b6cfa3bb0365e | graph_neighbor | ssrn-5231379.pdf | pages=[5, 6]


In [31]:
# Inspect the Knowledge Graph structure used for graph expansion

GRAPH_PATH = (
    VECTOR_STORE_DIR / "quant_research_knowledge_graph.graphml"
)

import networkx as nx

knowledge_graph = nx.read_graphml(GRAPH_PATH)

print("Graph nodes:", knowledge_graph.number_of_nodes())
print("Graph edges:", knowledge_graph.number_of_edges())

print("\nFirst 10 edges:")
for i, (u, v, data) in enumerate(knowledge_graph.edges(data=True)):
    print(u, "→", v, "|", data)
    if i >= 9:
        break

Graph nodes: 3426
Graph edges: 10111

First 10 edges:
document::doc_127ff1e6f5f2 → chunk::chunk_634b3e042dffad81 | {'relation': 'contains_chunk'}
document::doc_127ff1e6f5f2 → section::doc_127ff1e6f5f2::section::0::QUANTS AND MARKET EFFICIENCY | {'relation': 'contains_section'}
document::doc_127ff1e6f5f2 → chunk::chunk_e9dbc9860c200c4a | {'relation': 'contains_chunk'}
document::doc_127ff1e6f5f2 → section::doc_127ff1e6f5f2::section::0::1. Introduction | {'relation': 'contains_section'}
document::doc_127ff1e6f5f2 → chunk::chunk_f0d41995c19a42f8 | {'relation': 'contains_chunk'}
document::doc_127ff1e6f5f2 → chunk::chunk_09712835319a6dd6 | {'relation': 'contains_chunk'}
document::doc_127ff1e6f5f2 → chunk::chunk_d9600aaaec5dcae2 | {'relation': 'contains_chunk'}
document::doc_127ff1e6f5f2 → chunk::chunk_f1c8dbdb6d16fb1b | {'relation': 'contains_chunk'}
document::doc_127ff1e6f5f2 → chunk::chunk_424b6cfa3bb0365e | {'relation': 'contains_chunk'}
document::doc_127ff1e6f5f2 → chunk::chunk_005af1867

In [32]:
# Inspect chunk-to-chunk graph relationships

chunk_edges = []

for u, v, data in knowledge_graph.edges(data=True):
    if str(u).startswith("chunk::") and str(v).startswith("chunk::"):
        chunk_edges.append((u, v, data.get("relation")))

print("Chunk-to-chunk edges:", len(chunk_edges))

print("\nFirst 20 chunk relationships:")
for u, v, relation in chunk_edges[:20]:
    print(
        u.replace("chunk::", ""),
        "→",
        v.replace("chunk::", ""),
        "|",
        relation
    )

Chunk-to-chunk edges: 5550

First 20 chunk relationships:
chunk_634b3e042dffad81 → chunk_e9dbc9860c200c4a | next_chunk
chunk_e9dbc9860c200c4a → chunk_634b3e042dffad81 | previous_chunk
chunk_e9dbc9860c200c4a → chunk_f0d41995c19a42f8 | next_chunk
chunk_e9dbc9860c200c4a → chunk_f0d41995c19a42f8 | same_section
chunk_e9dbc9860c200c4a → chunk_09712835319a6dd6 | same_section
chunk_f0d41995c19a42f8 → chunk_e9dbc9860c200c4a | previous_chunk
chunk_f0d41995c19a42f8 → chunk_e9dbc9860c200c4a | same_section
chunk_f0d41995c19a42f8 → chunk_09712835319a6dd6 | next_chunk
chunk_f0d41995c19a42f8 → chunk_09712835319a6dd6 | same_section
chunk_f0d41995c19a42f8 → chunk_d9600aaaec5dcae2 | same_section
chunk_09712835319a6dd6 → chunk_f0d41995c19a42f8 | previous_chunk
chunk_09712835319a6dd6 → chunk_f0d41995c19a42f8 | same_section
chunk_09712835319a6dd6 → chunk_d9600aaaec5dcae2 | next_chunk
chunk_09712835319a6dd6 → chunk_d9600aaaec5dcae2 | same_section
chunk_09712835319a6dd6 → chunk_e9dbc9860c200c4a | same_section

In [34]:
test_chunk = "chunk_e9dbc9860c200c4a"
node = f"chunk::{test_chunk}"

print("Neighbors of:", test_chunk)
print("=" * 60)

for neighbor in knowledge_graph.neighbors(node):
    edge_data = knowledge_graph.get_edge_data(node, neighbor)

    print(
        neighbor.replace("chunk::", ""),
        "|",
        edge_data
    )

Neighbors of: chunk_e9dbc9860c200c4a
element::elem_70cbfd2a20c7 | {0: {'relation': 'contains_element'}}
element::elem_f5e945bfafb3 | {0: {'relation': 'contains_element'}}
chunk_634b3e042dffad81 | {0: {'relation': 'previous_chunk'}}
chunk_f0d41995c19a42f8 | {0: {'relation': 'next_chunk'}, 1: {'relation': 'same_section'}}
chunk_09712835319a6dd6 | {0: {'relation': 'same_section'}}


In [35]:
def get_graph_neighbors(chunk_id, graph):
    node = f"chunk::{chunk_id}"
    neighbors = []

    if node not in graph:
        return neighbors

    for neighbor in graph.neighbors(node):
        if not neighbor.startswith("chunk::"):
            continue

        edge_data = graph.get_edge_data(node, neighbor)

        relations = [
            data.get("relation")
            for data in edge_data.values()
        ]

        neighbors.append({
            "chunk_id": neighbor.replace("chunk::", ""),
            "relations": relations
        })

    return neighbors


# Test
neighbors = get_graph_neighbors(
    "chunk_e9dbc9860c200c4a",
    knowledge_graph
)

for item in neighbors:
    print(item)

{'chunk_id': 'chunk_634b3e042dffad81', 'relations': ['previous_chunk']}
{'chunk_id': 'chunk_f0d41995c19a42f8', 'relations': ['next_chunk', 'same_section']}
{'chunk_id': 'chunk_09712835319a6dd6', 'relations': ['same_section']}


In [36]:
test_reranked_ids = [
    "chunk_634b3e042dffad81",
    "chunk_e9dbc9860c200c4a",
    "chunk_f0d41995c19a42f8",
    "chunk_09712835319a6dd6",
    "chunk_d9600aaaec5dcae2",
]

for chunk_id in test_reranked_ids:
    neighbors = get_graph_neighbors(chunk_id, knowledge_graph)

    print(f"\n{chunk_id}")
    for item in neighbors:
        print(
            "  →",
            item["chunk_id"],
            "|",
            ", ".join(item["relations"])
        )


chunk_634b3e042dffad81
  → chunk_e9dbc9860c200c4a | next_chunk

chunk_e9dbc9860c200c4a
  → chunk_634b3e042dffad81 | previous_chunk
  → chunk_f0d41995c19a42f8 | next_chunk, same_section
  → chunk_09712835319a6dd6 | same_section

chunk_f0d41995c19a42f8
  → chunk_e9dbc9860c200c4a | previous_chunk, same_section
  → chunk_09712835319a6dd6 | next_chunk, same_section
  → chunk_d9600aaaec5dcae2 | same_section

chunk_09712835319a6dd6
  → chunk_f0d41995c19a42f8 | previous_chunk, same_section
  → chunk_d9600aaaec5dcae2 | next_chunk, same_section
  → chunk_e9dbc9860c200c4a | same_section
  → chunk_f1c8dbdb6d16fb1b | same_section

chunk_d9600aaaec5dcae2
  → chunk_09712835319a6dd6 | previous_chunk, same_section
  → chunk_f1c8dbdb6d16fb1b | next_chunk, same_section
  → chunk_f0d41995c19a42f8 | same_section
  → chunk_424b6cfa3bb0365e | same_section


In [37]:
def expand_with_graph(reranked_ids, graph):
    selected = list(reranked_ids)
    selected_set = set(selected)

    for chunk_id in reranked_ids:
        neighbors = get_graph_neighbors(chunk_id, graph)

        for item in neighbors:
            neighbor_id = item["chunk_id"]

            if neighbor_id not in selected_set:
                selected.append(neighbor_id)
                selected_set.add(neighbor_id)

    return selected


# Test on the current 5 reranked chunks
expanded_ids = expand_with_graph(
    test_reranked_ids,
    knowledge_graph
)

print("Reranked chunks:", len(test_reranked_ids))
print("Graph-aware context:", len(expanded_ids))
print("\nAdded graph chunks:")

for chunk_id in expanded_ids:
    if chunk_id not in test_reranked_ids:
        print("→", chunk_id)

Reranked chunks: 5
Graph-aware context: 7

Added graph chunks:
→ chunk_f1c8dbdb6d16fb1b
→ chunk_424b6cfa3bb0365e


In [38]:
reranked_rankings = []

for item in evaluation_set:
    query = item["query"]

    # Dense
    query_vector = embed_query(query)
    _, dense_indices = faiss_index.search(query_vector, 10)

    dense_ids = [
        chunk_metadata[idx]["chunk_id"]
        for idx in dense_indices[0]
    ]

    # BM25
    bm25_scores = bm25.get_scores(query.split())
    bm25_indices = np.argsort(bm25_scores)[::-1][:10]

    bm25_ids = [
        bm25_chunk_ids[idx]
        for idx in bm25_indices
    ]

    # RRF
    hybrid_ranked = rrf_fuse(dense_ids, bm25_ids)
    hybrid_ids = [x[0] for x in hybrid_ranked[:10]]

    # Cross-Encoder
    pairs = [
        [query, chunk_lookup[chunk_id]["retrieval_text"]]
        for chunk_id in hybrid_ids
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(hybrid_ids, scores),
        key=lambda x: x[1],
        reverse=True
    )

    reranked_ids = [x[0] for x in ranked[:5]]

    reranked_rankings.append({
        "query_id": item["query_id"],
        "reranked_ids": reranked_ids
    })

print("Stored reranked rankings:", len(reranked_rankings))
print(reranked_rankings[0])

Stored reranked rankings: 26
{'query_id': 'SEM01', 'reranked_ids': ['chunk_e9dbc9860c200c4a', 'chunk_f0d41995c19a42f8', 'chunk_005af18671e7dab4', 'chunk_a449942ec7fbfc65', 'chunk_fa626fbec884d89e']}


In [39]:
graph_results = []

for item, ranking in zip(evaluation_set, reranked_rankings):

    gold_ids = {
        x["chunk_id"]
        for x in item["relevant_chunks"]
    }

    graph_context_ids = expand_with_graph(
        ranking["reranked_ids"],
        knowledge_graph
    )

    covered = gold_ids.intersection(graph_context_ids)

    graph_results.append({
        "query_id": item["query_id"],
        "graph_coverage": len(covered) / len(gold_ids)
    })

graph_df = pd.DataFrame(graph_results)

print("Graph-aware coverage:", graph_df["graph_coverage"].mean())

Graph-aware coverage: 0.6538461538461539


In [40]:
graph_results = []

for item, ranking in zip(evaluation_set, reranked_rankings):

    gold_ids = {
        x["chunk_id"]
        for x in item["relevant_chunks"]
    }

    graph_context_ids = expand_with_graph(
        ranking["reranked_ids"],
        knowledge_graph
    )

    covered = gold_ids.intersection(graph_context_ids)

    graph_results.append({
        "query_id": item["query_id"],
        "graph_coverage": len(covered) / len(gold_ids)
    })

graph_df = pd.DataFrame(graph_results)

print(
    "Graph-aware coverage:",
    round(graph_df["graph_coverage"].mean(), 4)
)

Graph-aware coverage: 0.6538


“I evaluated the graph-aware context using the same frozen 26-question benchmark. For each query, I took the top-5 Cross-Encoder results and expanded them using direct Knowledge Graph relationships such as next_chunk, previous_chunk, and same_section. I then measured whether the gold relevant chunk was present in the expanded context. The graph-aware context achieved 65.38% coverage across the benchmark.”

In [41]:
RAG_RESPONSE_PATH = (
    VECTOR_STORE_DIR / "rag_last_response.json"
)

with open(RAG_RESPONSE_PATH, "r", encoding="utf-8") as f:
    rag_response = json.load(f)

print("RAG response keys:")
print(rag_response.keys())

print("\nQuery:")
print(rag_response["query"])

print("\nAnswer:")
print(rag_response["answer"])

print("\nCitations:")
print(rag_response["citations"])

RAG response keys:
dict_keys(['query', 'status', 'answer', 'citations', 'traceability', 'guardrail', 'llm_model'])

Query:
What does the retrieved research say about quantitative trading and market efficiency?

Answer:
Based on the provided research context, quantitative trading affects market efficiency depending on the quality of the quants' information and their level of participation:

* **Information Quality:** When quants' information quality is low, an increase in quantitative trading participation deteriorates price efficiency [S1, S5]. When their information quality is high, it improves price efficiency [S1, S5]. At moderate information quality levels, the relationship is U-shaped: price efficiency initially decreases, and then improves once the quantitative investing level exceeds a threshold [S5]. 
* **Underlying Mechanisms:** Quants are characterized by two opposing forces—superior information processing capacity (which increases trading aggressiveness) and limited strategi

In [42]:
traceability = rag_response["traceability"]

print("Citations:", rag_response["citations"])
print("Traceability records:", len(traceability))

for item in traceability:
    print(
        item["citation"],
        "→",
        item["chunk_id"],
        "→",
        item["source_file"],
        "→ pages",
        item["pages"]
    )

Citations: ['S5', 'S6']
Traceability records: 2
S5 → chunk_d9600aaaec5dcae2 → ssrn-5231379.pdf → pages [4]
S6 → chunk_f1c8dbdb6d16fb1b → ssrn-5231379.pdf → pages [5]


In [43]:
# Test RAG generation for all evaluation queries

rag_test_results = []

for item in evaluation_set[:2]:
    query = item["query"]

    print("=" * 70)
    print("Query:", item["query_id"])
    print(query)

    # Use the existing retrieval pipeline
    query_vector = embed_query(query)

    _, dense_indices = faiss_index.search(query_vector, 10)

    dense_ids = [
        chunk_metadata[idx]["chunk_id"]
        for idx in dense_indices[0]
    ]

    bm25_scores = bm25.get_scores(query.split())
    bm25_indices = np.argsort(bm25_scores)[::-1][:10]

    bm25_ids = [
        bm25_chunk_ids[idx]
        for idx in bm25_indices
    ]

    hybrid_ranked = rrf_fuse(dense_ids, bm25_ids)
    hybrid_ids = [x[0] for x in hybrid_ranked[:10]]

    pairs = [
        [query, chunk_lookup[chunk_id]["retrieval_text"]]
        for chunk_id in hybrid_ids
    ]

    ce_scores = reranker.predict(pairs)

    ranked = sorted(
        zip(hybrid_ids, ce_scores),
        key=lambda x: x[1],
        reverse=True
    )

    reranked_ids = [x[0] for x in ranked[:5]]

    graph_context_ids = expand_with_graph(
        reranked_ids,
        knowledge_graph
    )

    context = "\n\n".join(
        chunk_lookup[chunk_id]["retrieval_text"]
        for chunk_id in graph_context_ids
        if chunk_id in chunk_lookup
    )

    result = generate_grounded_answer(
        query,
        context
    )

    rag_test_results.append({
        "query_id": item["query_id"],
        "query": query,
        "answer": result.get("answer"),
        "citations": result.get("citations", []),
        "status": result.get("status")
    })

    print("Status:", result.get("status"))
    print("Citations:", result.get("citations"))

Query: SEM01
What developments have enabled the rise of quantitative investors (quants), and how do they differ from traditional discretionary investors?


NameError: name 'generate_grounded_answer' is not defined

In [44]:
[name for name in globals() if "answer" in name.lower() or "rag" in name.lower()]

['RAG_RESPONSE_PATH', 'rag_response', 'rag_test_results']

In [45]:
# 07 — Grounded RAG generation

def generate_grounded_answer(query: str, context: str):

    guardrail = check_input_guardrail(query)

    if guardrail["blocked"]:
        return {
            "status": "blocked",
            "answer": (
                "I can't process that instruction because it attempts to "
                "override the system instructions or request protected information."
            ),
            "guardrail": guardrail,
            "citations": [],
        }

    user_message = f"""
CONTEXT:
{context}

USER QUESTION:
{query}

Answer only from the supplied CONTEXT.
Use [S1], [S2], ... citations for substantive claims.
"""

    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=user_message,
        config=genai.types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1,
            max_output_tokens=900,
        ),
    )

    answer = (response.text or "").strip()

    cited_numbers = sorted(
    {int(x) for x in re.findall(r"\[S(\d+)\]", answer)}
    )

    return {
        "status": "success",
        "answer": answer,
        "guardrail": guardrail,
        "citations": [f"S{x}" for x in cited_numbers],
    }

In [46]:
item = evaluation_set[0]

query = item["query"]

print("Query:")
print(query)

print("\nExpected relevant chunks:")
print(item["relevant_chunks"])

Query:
What developments have enabled the rise of quantitative investors (quants), and how do they differ from traditional discretionary investors?

Expected relevant chunks:
[{'chunk_id': 'chunk_e9dbc9860c200c4a', 'relevance': 3}]


In [48]:
[name for name in globals() if any(x in name.lower() for x in ["search", "retrieve", "rerank"])]

['retrieved_ids',
 'bm25_retrieved_ids',
 'RERANKER_MODEL_NAME',
 'reranker',
 'rerank_results',
 'reranked_ids',
 'rerank_df',
 'rerank_summary',
 'comparison_rerank_df',
 'reranked_row',
 'test_reranked_ids',
 'reranked_rankings']